# Eating-duration eval: YOLO vs. YOLO + SAM 3

We want to know **when a meal starts and ends** from egocentric/table video. Two signals:

* **YOLO** (`unified-1-runpod/weights/best.pt`, 4 classes: `food`, `plate_empty`, `glass_full`, `glass_empty`), fast, every frame.
* **SAM 3** fill level: segment `food`, `bowl` and `hand`; fill = food area / visible bowl area. Slow (~0.5 s), run every 2 s.

Sections:
1. YOLO alone on the two eval images
2. SAM 3 fill level on the same images
3. Verdicts side by side
4. Full videos through `src/eater/monitor.py`: measured meal duration vs. a reference timeline
5. Speed

In [ ]:
import json, time
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ultralytics import YOLO

from eater.monitor import DEFAULT_SAM, DEFAULT_YOLO, FillMeter, MealTracker

ROOT = Path("/home/neckles/projects/nutriglass")
EVAL = {"empty": ROOT / "data/eval/empty.png", "full": ROOT / "data/eval/full.png"}  # expected state -> image
START_FILL, END_FILL = 0.30, 0.15  # same thresholds as monitor.py

yolo = YOLO(DEFAULT_YOLO)
meter = FillMeter(DEFAULT_SAM, dish_prompt="bowl")
print("YOLO weights:", DEFAULT_YOLO.relative_to(ROOT))
print("YOLO classes:", yolo.names)

## 1. YOLO alone

In [ ]:
yolo_rows, yolo_vis = [], {}
for expected, path in EVAL.items():
    r = yolo.predict(str(path), conf=0.01, device=0, verbose=False)[0]
    dets = sorted(zip(r.boxes.cls.tolist(), r.boxes.conf.tolist()), key=lambda d: -d[1])
    top = {yolo.names[int(c)]: round(s, 3) for c, s in dets[:3]}
    yolo_rows.append({"image": path.name, "expected": expected, "top detections (conf >= 0.01)": top})
    yolo_vis[expected] = yolo.predict(str(path), conf=0.25, device=0, verbose=False)[0].plot()[:, :, ::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, im) in zip(axes, yolo_vis.items()):
    ax.imshow(im); ax.set_title(f"YOLO, expected: {name}"); ax.axis("off")
plt.tight_layout(); plt.show()
pd.DataFrame(yolo_rows)

YOLO calls the scraped bowl `food`. Neither training set shows an empty *bowl*: the food dataset's
bowls are all labelled `food`, and the plates dataset's empty examples are flat plates. So it learned
"bowl means food".

## 2. SAM 3 fill level

In [ ]:
def overlay(frame, m):
    vis = frame.copy()
    vis[m["hand"]] = (0.6 * vis[m["hand"]] + 0.4 * np.array([255, 120, 0])).astype(np.uint8)
    vis[m["food"]] = (0.55 * vis[m["food"]] + 0.45 * np.array([0, 220, 0])).astype(np.uint8)
    cs, _ = cv2.findContours(m["dish"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(vis, cs, -1, (0, 200, 255), 4)
    return vis[:, :, ::-1]

def verdict(fill):
    if fill is None: return "no reading"
    return "full" if fill >= START_FILL else "empty" if fill <= END_FILL else "partly eaten"

sam_rows, sam_vis = [], {}
for expected, path in EVAL.items():
    frame = cv2.imread(str(path))
    meter.measure(frame)  # warm-up
    t = time.perf_counter(); m = meter.measure(frame); ms = (time.perf_counter() - t) * 1000
    sam_rows.append({"image": path.name, "expected": expected, "fill": m["fill"],
                     "visible bowl": m["visible_frac"], "SAM verdict": verdict(m["fill"]), "ms": round(ms)})
    sam_vis[expected] = overlay(frame, m)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, row in zip(axes, sam_rows):
    ax.imshow(sam_vis[row["expected"]]); ax.axis("off")
    ax.set_title(f"SAM 3, expected {row['expected']}: fill {row['fill']:.0%} -> {row['SAM verdict']}")
plt.tight_layout(); plt.show()
pd.DataFrame(sam_rows).style.format({"fill": "{:.1%}", "visible bowl": "{:.0%}"})

Green = food, orange outline = bowl (interior filled in), blue = hands (excluded from the bowl area).

## 3. Verdicts

In [ ]:
yolo_says = {r["expected"]: next(iter(r["top detections (conf >= 0.01)"]), "nothing") for r in yolo_rows}
to_state = {"food": "full", "plate_empty": "empty"}
verdicts = pd.DataFrame([{
    "image": r["image"], "expected": r["expected"],
    "YOLO": yolo_says[r["expected"]], "YOLO correct": to_state.get(yolo_says[r["expected"]]) == r["expected"],
    "SAM 3 fill": f"{r['fill']:.0%}", "SAM 3": r["SAM verdict"], "SAM 3 correct": r["SAM verdict"] == r["expected"],
} for r in sam_rows])
verdicts

## 4. Full videos

`src/eater/monitor.py` was run on each video beforehand (it takes several minutes per video):

```
uv run python src/eater/monitor.py <video> --out runs/monitor/<name>
```

**Reference timelines** were read by eye from contact sheets sampled every 5 s, so they're good to about ±5 s.
"Served" is when a full bowl is first on screen; "empty" is the first frame where only scraps remain.
The **salmon** video was never used for training; the **steak** video is the source of the two eval images.

In [ ]:
REFERENCE = {  # seconds, from 5 s contact sheets
    "steak":  {"served": 20, "empty": 545, "video": "ASMR： Eating a Steak & Potato Bowl (No talking) [xrA3diDmiyY].mkv"},
    "salmon": {"served": 10, "empty": 605, "video": "ASMR： Eating Salmon & Rice (No talking) [GHnZbdHRdMM].mp4"},
}

results = []
fig, axes = plt.subplots(len(REFERENCE), 1, figsize=(12, 3.6 * len(REFERENCE)), squeeze=False)
for ax, (name, ref) in zip(axes[:, 0], REFERENCE.items()):
    run = ROOT / "runs/monitor" / name
    if not (run / "summary.json").exists():
        ax.set_title(f"{name}: not run yet"); continue
    s = json.loads((run / "summary.json").read_text())
    tl = pd.read_csv(run / "timeline.csv")
    ax.plot(tl.t, tl.fill_raw, ".", color="0.65", ms=4, label="SAM fill (raw)")
    ax.plot(tl.t, tl.fill_smooth, "-", color="C0", label="smoothed")
    ax.axhline(START_FILL, color="C2", ls="--", lw=0.8); ax.axhline(END_FILL, color="C3", ls="--", lw=0.8)
    ax.axvline(ref["served"], color="k", ls=":", label="reference served / empty"); ax.axvline(ref["empty"], color="k", ls=":")
    if s["meal_start_s"] is not None:
        ax.axvspan(s["meal_start_s"], s["meal_end_s"] or tl.t.iloc[-1], color="C1", alpha=0.15, label="detected meal")
    # Where YOLO alone would say the bowl is empty
    yolo_empty = tl[tl.yolo_class == "plate_empty"]
    ax.plot(yolo_empty.t, np.full(len(yolo_empty), 0.97), "|", color="C3", ms=8, label="YOLO says plate_empty")
    ax.set_ylim(0, 1); ax.set_xlabel("time (s)"); ax.set_ylabel("food / visible bowl"); ax.set_title(name)
    ax.legend(loc="center left", fontsize=8)

    ref_dur = ref["empty"] - ref["served"]
    last_part = tl[tl.t >= ref["empty"]]
    results.append({
        "video": name,
        "reference start (s)": ref["served"], "detected start (s)": s["meal_start_s"],
        "reference end (s)": ref["empty"], "detected end (s)": s["meal_end_s"],
        "reference duration": f"{ref_dur // 60}:{ref_dur % 60:02d}",
        "detected duration": None if s["duration_s"] is None else f"{int(s['duration_s'] // 60)}:{int(s['duration_s'] % 60):02d}",
        "duration error (s)": None if s["duration_s"] is None else round(s["duration_s"] - ref_dur, 1),
        "YOLO says empty after reference end": f"{(last_part.yolo_class == 'plate_empty').mean():.0%}" if len(last_part) else "n/a",
        "processing time": f"{s['processing_s'] / 60:.1f} min",
    })
plt.tight_layout(); plt.show()
pd.DataFrame(results)

## 5. Speed

In [ ]:
import torch
frame = cv2.imread(str(EVAL["full"]))
t = time.perf_counter(); [yolo.predict(frame, device=0, verbose=False) for _ in range(20)]; yolo_ms = (time.perf_counter() - t) / 20 * 1000
t = time.perf_counter(); [meter.measure(frame) for _ in range(5)]; sam_ms = (time.perf_counter() - t) / 5 * 1000
pd.DataFrame([
    {"component": "YOLO (every analysed frame)", "ms per call": round(yolo_ms, 1), "max rate": f"{1000 / yolo_ms:.0f} fps"},
    {"component": "SAM 3 fill (food + bowl + hand)", "ms per call": round(sam_ms), "max rate": f"{1000 / sam_ms:.1f} fps"},
]).assign(gpu=torch.cuda.get_device_name())